# Evaluating anomaly detectors

This notebook answers the question: **if the trigger may keep only a small fraction of normal (zero-bias)
events, how much of each signal does each model catch?**

Every model you evaluate can be saved as a *trial* in `output/trials.csv`. At the end of the notebook,
all trials are compared in one plot of performance vs. model cost (how big and slow it would be on an FPGA).

<!-- TODO(organisers): explain trigger acceptance / rate and why we fix it. -->

In [ ]:
import json

import matplotlib.pyplot as plt

import plotting
from dataloader import create_datasets
from utils import (compute_scores, efficiency_at_acceptance, load_inputs, load_model, load_trials, model_cost, roc,
                   save_trial)

In [ ]:
# (trial name, path to a model saved by train.ipynb). The name is what the model is called in the trials file.
MODELS = [
    ("dense-ae", "output/dense-ae/best_model.pt"),
    ("dense-qae", "output/dense-qae/best_model.pt"),
]
CONFIG = "config/baseline.yml"  # the test samples come from this config

# Fraction of zero-bias events the trigger may keep.
# TODO(organisers): choose the operating point for the hackathon.
ACCEPTANCE = 1e-3

In [ ]:
test_sets = {}  # test datasets, loaded once for each different input choice
scores = {}     # scores[model][sample] = anomaly scores
cost = {}       # cost[model] = {"n_params": ..., "bops": ..., "n_layers": ...}, see utils.model_cost

for model_name, model_path in MODELS:
    model = load_model(model_path)
    inputs = load_inputs(model_path)  # the layout, features and input type the model was trained with
    key = json.dumps([inputs["layout"], inputs["features"], inputs["input_type"]])
    if key not in test_sets:
        _, _, test_sets[key] = create_datasets(CONFIG, layout=inputs["layout"], features=inputs["features"],
                                               input_type=inputs["input_type"], load_test=True, load_train=False)
    scores[model_name] = {name: compute_scores(model, data) for name, data in test_sets[key].items()}
    cost[model_name] = model_cost(model, test_sets[key]["background"].x.shape[1:])
    print(f"{model_name}: {cost[model_name]}")

In [ ]:
metrics = {}       # metrics[model][signal] = {"auc": ..., "efficiency": ...}
efficiencies = {}  # efficiencies[model][signal] = efficiency, for the bar chart
for model_name, s in scores.items():
    rocs, metrics[model_name] = {}, {}
    for signal in s:
        if signal == "background":
            continue
        rocs[signal] = roc(s["background"], s[signal])
        eff = efficiency_at_acceptance(s["background"], s[signal], ACCEPTANCE)
        metrics[model_name][signal] = {"auc": rocs[signal][2], "efficiency": eff}
        print(f"{model_name:15s} {signal:35s} AUC = {rocs[signal][2]:.3f}   efficiency @ {ACCEPTANCE:g} = {eff:.4f}")
    efficiencies[model_name] = {sig: m["efficiency"] for sig, m in metrics[model_name].items()}

    plotting.plot_score_distributions(s, title=model_name)
    plotting.plot_roc(rocs, ACCEPTANCE, title=model_name)

In [ ]:
plotting.plot_signal_efficiency(efficiencies, ACCEPTANCE);

## Saving trials

Each model above is added to `output/trials.csv` (one row per signal), together with its cost
(number of parameters, bit operations and number of layers), the data config, the acceptance and an optional note. Saving a trial under a name that
already exists replaces it, so re-running this notebook doesn't create duplicates. Use a new name for every
variant you want to keep.

The file is plain CSV: share it with your team, or merge your teammates' trials into yours.
Use `utils.delete_trial("name")` to remove a trial.

In [ ]:
# Optional: a short description of each trial, e.g. {"dense-ae": "baseline, 5 epochs"}
NOTES = {}

for model_name, model_path in MODELS:
    trials = save_trial(
        model_name, metrics[model_name], cost[model_name],
        model_path=model_path, config=CONFIG, acceptance=ACCEPTANCE, notes=NOTES.get(model_name, ""),
    )

## All trials: performance vs. model cost

A trigger model has to be good **and** cheap: it must fit on the FPGA next to the other trigger algorithms, and decide
within the latency budget. Without building the firmware, we estimate the cost of a model with `utils.model_cost`:

- **`bops`**, bit operations: roughly how much FPGA area (logic) the model needs. Every multiplication of a `b_a`-bit
  input by a `b_w`-bit weight costs about `b_a * b_w`, so there are two ways down: fewer multiplications (fewer
  inputs, narrower layers) and fewer bits (quantization-aware training with Brevitas, see `dense-qae`). Weights that
  are exactly zero (pruned) are free. Unquantized layers count as 16 bits.
- **`n_params`**: the number of trainable parameters, the memory the weights need.
- **`n_layers`**: the number of dense layers an event goes through. Each layer adds a few clock cycles, so this is
  a rough measure of the latency.

The plot below shows every saved trial against its BOPs, with the metrics averaged over the signals. The dashed
line is the *Pareto front*: the trials that no other trial beats at a lower cost. Good models push the front up
and to the left. Use `cost="n_params"` or `cost="n_layers"` to compare against the other costs.

Only compare trials evaluated with the same config and acceptance (check the table).

In [ ]:
trials = load_trials()
summary = (trials.groupby("trial")
           .agg(bops=("bops", "first"), n_params=("n_params", "first"), n_layers=("n_layers", "first"),
                mean_auc=("auc", "mean"), mean_efficiency=("efficiency", "mean"),
                config=("config", "first"), acceptance=("acceptance", "first"), notes=("notes", "first"))
           .sort_values("bops"))
summary

In [ ]:
plotting.plot_trials(trials, outpath="output/trials.png");

In [ ]:
# The same for each signal separately: which signals benefit from a bigger model?
for signal in trials["signal"].unique():
    plotting.plot_trials(trials, signal=signal);

## Is the average what we care about?

A model can have a good average because it is excellent on one easy signal and blind to the others. Make a plot of the
**worst** signal efficiency of each trial against its BOPs. Does the best model change?

Hint: `trials` has one row per (trial, signal); `trials.groupby("trial")` and `.agg(...)` with `"min"` give you one row per trial.

In [ ]:
# TODO: your plot here

## Discussion

<!-- TODO(organisers): questions. Which signals are easy or hard to find, and why? How does the model compare to the existing trigger (L1bit)? -->